# Pattern Mining

## Präambel

Am Anfang eines Skripts werden meist alle Softwarepakete geladen, die darin verwendet werden. In diesem Skript machen wir eine Ausnahme, um uns schrittweise an die Logik von Python zu gewöhnen.

Zunächst laden wir das Paket `pandas`, das wir für die Bearbeitung von Datensätzen verwenden.

In [1]:
# Lade das Paket pandas und vergib den Alias pd dafür (das erspart den Tippaufwand – echt!)
import pandas as pd

## Daten

Wir verwenden einen Datensatz von Kaggle: https://www.kaggle.com/datasets/lissetteg/ecommerce-dataset

Kaggle (https://www.kaggle.com/) ist eine Plattform, über die Datensätze für die Entwicklung von Modellen zur Verfügung gestellt werden. Die Webseite ist vor allem als Host von Wettbewerben mit z. T. bemerkenswert hohen Preisgeldern bekannt. Die Modelle werden auch geteilt.

Um den Datensatz direkt in Python zu laden, verwenden wir das Paket `kagglehub`:

In [2]:
# Lade das Paket kagglehub, aber diesmal ohne Alias
import kagglehub

Der Import erzeugt keine Ausgabe. Falls hier `ModuleNotFoundError` erscheint, ist das Paket noch nicht installiert und kann mit `pip install kagglehub` nachinstalliert werden.

In [3]:
# Lade Version 1 des Datensatzes herunter (eine feste Version macht die Ergebnisse reproduzierbar)
path = kagglehub.dataset_download("lissetteg/ecommerce-dataset/versions/1") # Pfad zum heruntergeladenen Datensatz

# Verwende die Funktion read_csv aus dem pandas-Paket
df = pd.read_csv(path + '/data-2.csv')

100%|██████████| 7.20M/7.20M [00:02<00:00, 3.70MB/s]

Extracting files...


`kagglehub` lädt den Datensatz beim ersten Aufruf herunter (Fortschrittsbalken) und speichert ihn lokal zwischen. Bei späteren Aufrufen wird die lokale Kopie verwendet. Falls eine Warnung erscheint, dass eine neuere Version von `kagglehub` verfügbar ist, kann sie ignoriert werden.

In [4]:
# Wirf einen Blick auf den eingelesenen Datensatz
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


Jede Zeile ist eine Position auf einer Rechnung eines britischen Online-Händlers für Geschenkartikel (Dezember 2010 bis Dezember 2011):

* `InvoiceNo`: Rechnungsnummer. Eine Rechnung entspricht einem Einkauf (einer *Transaktion*) und kann viele Zeilen haben.
* `StockCode`, `Description`: Artikelnummer und Bezeichnung des Produkts
* `Quantity`, `UnitPrice`: Menge und Stückpreis
* `InvoiceDate`, `CustomerID`, `Country`: Zeitpunkt, Kundennummer und Land

Der Datensatz hat 541.909 Zeilen. Die Rechnung 536365 enthält zum Beispiel mehrere verschiedene Produkte. Das ist genau die Struktur, die wir für das Pattern Mining brauchen.

In [5]:
# Verteilungsmaße für die numerischen Spalten im Datensatz
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


`describe()` zeigt Kennzahlen der numerischen Spalten. Auffällig sind:

* **Negative Werte** bei `Quantity` (Minimum −80.995) und `UnitPrice` (Minimum −11.062): Das sind Stornierungen und Korrekturen, keine echten Käufe.
* **Extreme Ausreißer:** Das Maximum von `Quantity` (80.995) entspricht dem Betrag nach genau dem Minimum (−80.995), hier wurde offenbar eine Fehlbestellung storniert.
* **Fehlende Werte** bei `CustomerID`: Nur 406.829 von 541.909 Zeilen haben eine Kundennummer (`count`).
* Der Median der Menge ist 3, der Mittelwert 9,6: Die meisten Positionen sind klein, wenige sehr große ziehen den Mittelwert nach oben.

Für unsere Analyse sind vor allem die Stornierungen störend, die wir im Folgenden entfernen. Fehlende Kundennummern stören nicht, weil wir nur mit Rechnungen und Produkten arbeiten.

## Datenaufbereitung

Entferne Stornierungen und Buchungskorrekturen. Sie sind an einem „C“ bzw. „A“ in der Rechnungsnummer zu erkennen.

In [6]:
# Behalte nur Rechnungen, deren Nummer weder 'A' noch 'C' enthält (~ bedeutet 'nicht')
df = df[~df['InvoiceNo'].str.contains('A|C')]
len(df) # Ergibt jetzt 532618

532618

Rechnungsnummern mit „C“ sind Stornierungen (*cancelled*), solche mit „A“ Buchungskorrekturen (*adjust bad debt*). Nach dem Entfernen bleiben 532.618 Zeilen übrig, also wurden gut 9.000 Zeilen gelöscht.

Betrachte nur Daten aus Dänemark, Deutschland und Italien.

In [7]:
# Wähle Daten aus drei Ländern
subset_df = df[df['Country'].isin(['Denmark', 'Germany', 'Italy'])]
subset_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1109,536527,22809,SET OF 6 T-LIGHTS SANTA,6,12/1/2010 13:04,2.95,12662.0,Germany
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,12/1/2010 13:04,2.55,12662.0,Germany
1111,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,12/1/2010 13:04,0.85,12662.0,Germany
1112,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,12/1/2010 13:04,1.65,12662.0,Germany
1113,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,12/1/2010 13:04,1.95,12662.0,Germany
...,...,...,...,...,...,...,...,...
541801,581578,22993,SET OF 4 PANTRY JELLY MOULDS,12,12/9/2011 12:16,1.25,12713.0,Germany
541802,581578,22907,PACK OF 20 NAPKINS PANTRY DESIGN,12,12/9/2011 12:16,0.85,12713.0,Germany
541803,581578,22908,PACK OF 20 NAPKINS RED APPLES,12,12/9/2011 12:16,0.85,12713.0,Germany
541804,581578,23215,JINGLE BELL HEART ANTIQUE SILVER,12,12/9/2011 12:16,2.08,12713.0,Germany


Übrig bleiben nur die Rechnungspositionen aus Dänemark, Deutschland und Italien. Die Zahlen links sind weiterhin der Index aus dem ursprünglichen Datensatz.

Der Apriori-Algorithmus erwartet die Daten in einem speziellen Format: eine Zeile pro Rechnung und eine Spalte pro Produkt. Zunächst summieren wir pro Rechnung und Produkt die gekauften Mengen:

In [8]:
# Summiere die Menge je Kombination aus Rechnung und Produkt
subset_df = subset_df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum()
subset_df

InvoiceNo  Description                        
536527     3 HOOK HANGER MAGIC GARDEN             12
           5 HOOK HANGER MAGIC TOADSTOOL          12
           5 HOOK HANGER RED MAGIC TOADSTOOL      12
           ASSORTED COLOUR LIZARD SUCTION HOOK    24
           CHILDREN'S CIRCUS PARADE MUG           12
                                                  ..
581578     SPOTTY BUNTING                          9
           VINTAGE DONKEY TAIL GAME                6
           WRAP ALPHABET POSTER                   25
           WRAP CIRCUS PARADE                     25
           WRAP RED APPLES                        25
Name: Quantity, Length: 10151, dtype: int64

Das Ergebnis ist eine `Series` mit einem zweistufigen Index (Rechnung und Produkt). Für jede Kombination steht die gekaufte Menge. Insgesamt gibt es 10.151 solcher Kombinationen.

Bringe die Daten in ein „breites“ Format, in dem für jedes Produkt eine eigene Spalte mit der pro Rechnung bestellten Menge enthalten ist.

In [9]:
# Eine Zeile pro Rechnung
# Wenn ein Produkt in einer Rechnung enthalten war, steht dort ein Wert, ansonsten NaN
subset_df = subset_df.unstack()
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


`unstack()` macht aus der zweiten Indexstufe (`Description`) Spalten. Jetzt gibt es eine Zeile pro Rechnung und eine Spalte pro Produkt. Die meisten Felder sind `NaN`, weil eine Rechnung nur einen kleinen Teil aller Produkte enthält.

Ersetze die `NaN`-Werte durch 0. Im nächsten Schritt prüfen wir dann einheitlich, ob die Menge größer als 0 ist.

In [10]:
# Ersetze fehlende Werte (Produkt nicht gekauft) durch 0
subset_df = subset_df.fillna(0)
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536967,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581494,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Statt `NaN` steht jetzt 0, wenn ein Produkt in einer Rechnung nicht vorkommt.

Die Tabelle hat 513 Zeilen (Rechnungen) und 1.787 Spalten (Produkte).

Lösche die Spalte `POSTAGE` (Versandkosten).

In [11]:
# Versandkosten sind kein Produkt und würden die Ergebnisse verzerren
subset_df = subset_df.drop("POSTAGE", axis=1)

Im letzten Schritt wandeln wir die Mengen in boolesche Werte um: `True`, wenn die Menge größer als 0 ist, sonst `False`.

In [12]:
# True, wenn das Produkt in der Rechnung gekauft wurde (Menge > 0), sonst False
subset_df = subset_df > 0

# Zeige Ergebnis
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536840,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536861,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536967,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536983,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581494,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581570,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Jetzt steht in jeder Zelle nur noch, *ob* ein Produkt in der Rechnung enthalten war (`True`), aber nicht mehr, wie viel davon gekauft wurde. Dieses Format („One-Hot-Encoding“) erwartet `mlxtend`. Das Ergebnis ist ein Warenkorb pro Rechnung.

Das Resultat ist eine Tabelle mit 513 Zeilen und 1.786 Spalten:
* Der Zeilenindex (`InvoiceNo`) enthält die Rechnungsnummer, die als Transaktionsnummer dient.
* Die Spaltennamen (Achsenname `Description`) enthalten die Namen der Produkte.

Dieser Datensatz kann nun mithilfe vorprogrammierter Algorithmen analysiert werden.

## Apriori-Algorithmus

### Laden des Pakets

Das Paket `mlxtend` (https://rasbt.github.io/mlxtend/) enthält Funktionen, die den Apriori-Algorithmus implementieren.

Wir laden aus dem Modul `frequent_patterns` des Pakets die Funktionen `apriori` und `association_rules`:

In [13]:
# Lade die Funktionen für den Apriori-Algorithmus und die Association Rules
from mlxtend.frequent_patterns import apriori, association_rules

### Schritt 1: Finde häufige Itemsets

Wir setzen den minimalen Support auf 5 %. D. h. die Itemsets müssen in mindestens 5 % aller Rechnungen enthalten sein.

In [14]:
# Finde alle Itemsets, die in mindestens 5 % der Rechnungen vorkommen
frq_items = apriori(subset_df, min_support=0.05, use_colnames=True)

Die Zelle erzeugt keine Ausgabe, das Ergebnis ist in `frq_items` gespeichert.

Zeige das Ergebnis geordnet nach Support.

In [15]:
# Sortiere absteigend nach Support
frq_items.sort_values('support', ascending=False)

,support,itemsets
37,0.237817,(ROUND SNACK BOXES SET OF4 WOODLAND )
36,0.150097,(ROUND SNACK BOXES SET OF 4 FRUITS )
26,0.134503,(PLASTERS IN TIN WOODLAND ANIMALS)
33,0.132554,(REGENCY CAKESTAND 3 TIER)
23,0.124756,(PLASTERS IN TIN CIRCUS PARADE )
56,0.122807,"(ROUND SNACK BOXES SET OF4 WOODLAND , ROUND SN..."
47,0.118908,(WOODLAND CHARLOTTE BAG)
45,0.109162,(SPACEBOY LUNCH BOX )
24,0.103314,(PLASTERS IN TIN SPACEBOY)
14,0.097466,(JUMBO BAG WOODLAND ANIMALS)


Das Ergebnis enthält 59 Itemsets. `support` ist der Anteil der Rechnungen, die das Itemset enthalten:

* Am häufigsten ist die Snackbox „ROUND SNACK BOXES SET OF4 WOODLAND“: Sie steckt in 23,8 % aller 513 Rechnungen.
* Die meisten Itemsets bestehen aus einem einzigen Produkt. Itemsets mit zwei Produkten, etwa die beiden Snackbox-Varianten zusammen (12,3 %), sind seltener, weil *beide* Produkte gekauft werden müssen.
* Ein Itemset kann nie einen höheren Support haben als eine seiner Teilmengen, also auch nicht als sein seltenstes Produkt. Apriori nutzt das in umgekehrter Richtung: Ist ein Itemset selten, sind auch alle seine Obermengen selten und müssen gar nicht erst gezählt werden.

### Schritt 2: Finde starke Association Rules

Die Funktion `association_rules` bildet aus den häufigen Itemsets des vorherigen Schritts (Support ≥ 5 %) Regeln und berechnet für jede Regel u. a. Confidence und Lift. Wir behalten nur Regeln, deren Confidence mindestens 40 % beträgt.

In [16]:
# Berechne die Association Rules
rules = association_rules(frq_items, metric = 'confidence', min_threshold = .4)

# Zeige die wichtigsten Spalten geordnet nach 'confidence'
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False)

,antecedents,consequents,support,confidence,lift
9,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.058480,0.857143,7.208431
12,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.122807,0.818182,3.440387
1,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.052632,0.729730,5.849240
13,(SPACEBOY LUNCH BOX ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.068226,0.625000,2.628074
3,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.072125,0.578125,4.298234
8,(PLASTERS IN TIN WOODLAND ANIMALS),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.072125,0.536232,2.254811
2,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.536232,4.298234
7,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.054581,0.528302,3.927810
11,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.122807,0.516393,3.440387
14,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.060429,0.508197,2.136926


Jede Zeile ist eine Regel „wer `antecedents` kauft, kauft auch `consequents`“. Die wichtigsten Spalten:

* `support`: Anteil der Rechnungen, die beide Seiten der Regel enthalten
* `confidence`: Anteil der Rechnungen mit `antecedents`, die auch `consequents` enthalten (eine bedingte Wahrscheinlichkeit)
* `lift`: Verhältnis von `confidence` zum `consequent support`. Ein Wert über 1 bedeutet, dass die Produkte häufiger gemeinsam gekauft werden, als bei Unabhängigkeit zu erwarten wäre.

Die stärkste Regel lautet: Wer die „RED RETROSPOT CHARLOTTE BAG“ kauft, kauft in 85,7 % der Fälle auch die „WOODLAND CHARLOTTE BAG“. Ebenso enthalten 81,8 % der Rechnungen mit der Snackbox im Früchte-Design auch die Snackbox im Woodland-Design.

Die Regeln gelten nicht symmetrisch: Umgekehrt enthalten nur 49,2 % der Rechnungen mit der „WOODLAND CHARLOTTE BAG“ auch die rote Variante. Der Grund ist, dass die Woodland-Tasche insgesamt deutlich häufiger gekauft wird (11,9 % gegenüber 6,8 % der Rechnungen).

In [17]:
# Zeige die Ergebnisse geordnet nach 'lift'
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)

,antecedents,consequents,support,confidence,lift
10,(WOODLAND CHARLOTTE BAG),(RED RETROSPOT CHARLOTTE BAG),0.058480,0.491803,7.208431
9,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.058480,0.857143,7.208431
0,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN STRONGMAN),0.052632,0.421875,5.849240
1,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.052632,0.729730,5.849240
2,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.536232,4.298234
3,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.072125,0.578125,4.298234
6,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN SPACEBOY),0.054581,0.405797,3.927810
7,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.054581,0.528302,3.927810
11,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.122807,0.516393,3.440387
12,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.122807,0.818182,3.440387


Nach `lift` sortiert ergibt sich ein anderes Bild. Ganz oben stehen jetzt die beiden Charlotte Bags (Lift 7,2) und die Pflaster-Dosen „CIRCUS PARADE“ und „STRONGMAN“ (Lift 5,8): Diese Produkte werden rund siebenmal bzw. knapp sechsmal so häufig zusammen gekauft, wie es bei Unabhängigkeit zu erwarten wäre.

Der Lift ist für eine Regel und ihre Umkehrung immer gleich. Deshalb stehen beide Richtungen direkt untereinander, sofern auch die Umkehrung die Mindest-Confidence von 40 % erreicht. Bei fünf Regeln ist das nicht der Fall, etwa bei „SPACEBOY LUNCH BOX → ROUND SNACK BOXES SET OF4 WOODLAND“. Regeln mit der sehr beliebten Woodland-Snackbox als `consequents` haben zwar eine hohe Confidence, aber meist einen niedrigen Lift (2,1 bis 2,6), weil diese Snackbox ohnehin in fast jeder vierten Rechnung steckt. Eine Ausnahme ist die Früchte-Snackbox (Lift 3,4).

Inhaltlich zeigen die Regeln, dass Kundinnen und Kunden gerne mehrere Varianten desselben Produkts kaufen (Taschen, Pflaster-Dosen und Snackboxen in verschiedenen Designs).

### Schritt 3: Wie verlässlich sind die Regeln?

Die Confidence einer Regel ist ein Anteil, der aus einer Stichprobe von Rechnungen geschätzt wird. Wie verlässlich diese Schätzung ist, hängt davon ab, auf wie vielen Rechnungen sie beruht: 30 Treffer in 35 Rechnungen sind ein festerer Befund als 3 Treffer in 3 Rechnungen, obwohl die zweite Regel die höhere Confidence hat (100 % gegenüber 86 %). Wir prüfen die Regeln deshalb auf zwei Arten:

1. **Konfidenzintervall:** Wir berechnen für jede Confidence die untere Grenze eines 95-%-Konfidenzintervalls (*Wilson-Intervall*). Sie gibt an, welche Confidence die Regel mit hoher Sicherheit mindestens hat. Je weniger Rechnungen eine Regel stützen, desto weiter liegt die Grenze unter der Confidence.
2. **Stabilität über die Zeit:** Wir teilen die Rechnungen in eine erste und eine zweite Hälfte des Jahres und prüfen, ob die Regel in beiden Hälften gilt. Ein Muster, das nur in einem Zeitraum auftritt, taugt wenig für Vorhersagen.

In [18]:
import numpy as np

# Untere Grenze des 95-%-Konfidenzintervalls für einen Anteil k von n (Wilson-Intervall)
def wilson_lower(k, n, z=1.96):
    p = k / n
    center = p + z**2 / (2 * n)
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (center - margin) / (1 + z**2 / n)

# Zähle, wie viele der angegebenen Rechnungen die antecedents (n_A) bzw. die ganze Regel (n_AC) enthalten
def count_rule(rule, invoices):
    has_a = subset_df.loc[invoices, list(rule['antecedents'])].all(axis=1)
    has_ac = has_a & subset_df.loc[invoices, list(rule['consequents'])].all(axis=1)
    return has_a.sum(), has_ac.sum()

# Arbeite mit einer Kopie, damit die Tabelle rules unverändert bleibt
reliability = rules[['antecedents', 'consequents', 'confidence', 'lift']].copy()
counts = [count_rule(rule, subset_df.index) for _, rule in rules.iterrows()]
reliability['n_A'] = [n_a for n_a, n_ac in counts]
reliability['n_AC'] = [n_ac for n_a, n_ac in counts]
reliability['conf_low95'] = wilson_lower(reliability['n_AC'], reliability['n_A'])

# Sortiere nach der unteren Grenze der Confidence
reliability = reliability.sort_values('conf_low95', ascending=False)
reliability[['antecedents', 'consequents', 'n_A', 'n_AC', 'confidence', 'conf_low95', 'lift']]

,antecedents,consequents,n_A,n_AC,confidence,conf_low95,lift
12,frozenset({ROUND SNACK BOXES SET OF 4 FRUITS }),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),77,63,0.818182,0.717635,3.440387
9,frozenset({RED RETROSPOT CHARLOTTE BAG}),frozenset({WOODLAND CHARLOTTE BAG}),35,30,0.857143,0.706241,7.208431
1,frozenset({PLASTERS IN TIN STRONGMAN}),frozenset({PLASTERS IN TIN CIRCUS PARADE }),37,27,0.729730,0.570215,5.849240
13,frozenset({SPACEBOY LUNCH BOX }),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),56,35,0.625000,0.494051,2.628074
2,frozenset({PLASTERS IN TIN CIRCUS PARADE }),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),64,37,0.578125,0.456098,4.298234
11,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),frozenset({ROUND SNACK BOXES SET OF 4 FRUITS }),122,63,0.516393,0.428578,3.440387
3,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),frozenset({PLASTERS IN TIN CIRCUS PARADE }),69,37,0.536232,0.419782,4.298234
8,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),69,37,0.536232,0.419782,2.254811
6,frozenset({PLASTERS IN TIN SPACEBOY}),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),53,28,0.528302,0.396599,3.927810
14,frozenset({WOODLAND CHARLOTTE BAG}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),61,31,0.508197,0.386024,2.136926


`n_A` ist die Zahl der Rechnungen mit den `antecedents`, `n_AC` die Zahl der Rechnungen, die zusätzlich die `consequents` enthalten. Die Confidence ist also `n_AC / n_A`. `conf_low95` ist die untere Grenze des Konfidenzintervalls.

An der Spitze stehen dieselben zwei Regeln wie bei der Confidence, aber in umgekehrter Reihenfolge: Die Regel „ROUND SNACK BOXES SET OF 4 FRUITS → ROUND SNACK BOXES SET OF4 WOODLAND“ hat zwar die etwas niedrigere Confidence (81,8 % gegenüber 85,7 %), beruht aber auf 77 statt 35 Rechnungen. Ihre untere Grenze liegt deshalb höher (71,8 % gegenüber 70,6 %). Am Ende der Liste liegt die untere Grenze nur noch bei rund 30 %: Dort ist nicht gesichert, dass die Regel die Mindest-Confidence von 40 % tatsächlich erreicht.

Für die zweite Prüfung brauchen wir das Datum jeder Rechnung. Wir teilen die Rechnungen am Median des Datums, sodass beide Hälften gleich viele Rechnungen enthalten.

In [19]:
# Datum jeder Rechnung (alle Zeilen einer Rechnung haben dasselbe Datum)
invoice_dates = pd.to_datetime(df.groupby('InvoiceNo')['InvoiceDate'].first(), format='%m/%d/%Y %H:%M')
invoice_dates = invoice_dates.loc[subset_df.index] # Nur die Rechnungen aus subset_df

# Teile die Rechnungen am Median des Datums
split_date = invoice_dates.median()
first_half = subset_df.index[invoice_dates < split_date]
second_half = subset_df.index[invoice_dates >= split_date]
print(f"Grenze: {split_date:%d.%m.%Y}, erste Hälfte: {len(first_half)} Rechnungen, zweite Hälfte: {len(second_half)} Rechnungen")

# Berechne die Confidence jeder Regel getrennt für beide Hälften
for name, invoices in [('conf_H1', first_half), ('conf_H2', second_half)]:
    counts = [count_rule(rule, invoices) for _, rule in reliability.iterrows()]
    reliability[name] = [n_ac / n_a for n_a, n_ac in counts]

# Stabil: Die Regel erreicht die Mindest-Confidence von 40 % in beiden Hälften
reliability['stable'] = reliability[['conf_H1', 'conf_H2']].min(axis=1) >= 0.4

reliability[['antecedents', 'consequents', 'n_A', 'confidence', 'conf_low95', 'conf_H1', 'conf_H2', 'stable']]

Grenze: 27.07.2011, erste Hälfte: 256 Rechnungen, zweite Hälfte: 257 Rechnungen


,antecedents,consequents,n_A,confidence,conf_low95,conf_H1,conf_H2,stable
12,frozenset({ROUND SNACK BOXES SET OF 4 FRUITS }),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),77,0.818182,0.717635,0.846154,0.760000,True
9,frozenset({RED RETROSPOT CHARLOTTE BAG}),frozenset({WOODLAND CHARLOTTE BAG}),35,0.857143,0.706241,0.857143,0.857143,True
1,frozenset({PLASTERS IN TIN STRONGMAN}),frozenset({PLASTERS IN TIN CIRCUS PARADE }),37,0.729730,0.570215,0.700000,0.764706,True
13,frozenset({SPACEBOY LUNCH BOX }),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),56,0.625000,0.494051,0.636364,0.608696,True
2,frozenset({PLASTERS IN TIN CIRCUS PARADE }),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),64,0.578125,0.456098,0.583333,0.571429,True
11,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),frozenset({ROUND SNACK BOXES SET OF 4 FRUITS }),122,0.516393,0.428578,0.619718,0.372549,False
3,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),frozenset({PLASTERS IN TIN CIRCUS PARADE }),69,0.536232,0.419782,0.538462,0.533333,True
8,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),69,0.536232,0.419782,0.615385,0.433333,True
6,frozenset({PLASTERS IN TIN SPACEBOY}),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),53,0.528302,0.396599,0.518519,0.538462,True
14,frozenset({WOODLAND CHARLOTTE BAG}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),61,0.508197,0.386024,0.589744,0.363636,False


`conf_H1` und `conf_H2` sind die Confidence in der ersten und der zweiten Jahreshälfte. Die fünf verlässlichsten Regeln aus dem Konfidenzintervall halten auch über die Zeit: Die beiden Charlotte Bags kommen in beiden Hälften auf 85,7 %, die Snackboxen auf 84,6 % und 76,0 %, die Pflaster-Dosen „STRONGMAN → CIRCUS PARADE“ auf 70,0 % und 76,5 %.

Sechs der 15 Regeln verfehlen dagegen in einer Hälfte die Mindest-Confidence von 40 %. Auffällig sind Regeln mit der beliebten Woodland-Snackbox als `consequents`: „PLASTERS IN TIN CIRCUS PARADE → ROUND SNACK BOXES SET OF4 WOODLAND“ fällt von 63,9 % in der ersten auf 32,1 % in der zweiten Hälfte. Die Umkehrung der Snackbox-Regel („WOODLAND → FRUITS“) sinkt von 62,0 % auf 37,3 %.

Insgesamt sind die Regeln zwischen Varianten desselben Produkts (zwei Charlotte Bags, zwei Snackboxen, zwei Pflaster-Dosen) am verlässlichsten. Regeln, die verschiedene Produkte verbinden, hängen stärker davon ab, welche Produkte gerade im Angebot oder in Mode sind. Bevor man eine Regel nutzt, etwa für Produktempfehlungen, lohnt sich deshalb ein Blick auf die Zahl der Rechnungen dahinter und auf ihre Stabilität über die Zeit.

## FP-Growth-Algorithmus

Lade die Funktion `fpgrowth` aus dem Modul `frequent_patterns` des Pakets `mlxtend`:

In [20]:
# Lade die Funktion für den FP-Growth-Algorithmus
from mlxtend.frequent_patterns import fpgrowth

Wende den FP-Growth-Algorithmus wie oben an. Damit wir die Ergebnisse vergleichen können, speichern wir sie unter neuen Namen (`frq_items_fp` und `rules_fp`).

### Schritt 1: Finde häufige Itemsets

In [21]:
# Finde alle Itemsets mit mindestens 5 % Support, diesmal mit FP-Growth
frq_items_fp = fpgrowth(subset_df, min_support=0.05, use_colnames=True)

# Zeige Ergebnis sortiert nach Support an
frq_items_fp.sort_values('support', ascending=False)

,support,itemsets
9,0.237817,(ROUND SNACK BOXES SET OF4 WOODLAND )
10,0.150097,(ROUND SNACK BOXES SET OF 4 FRUITS )
20,0.134503,(PLASTERS IN TIN WOODLAND ANIMALS)
11,0.132554,(REGENCY CAKESTAND 3 TIER)
12,0.124756,(PLASTERS IN TIN CIRCUS PARADE )
49,0.122807,"(ROUND SNACK BOXES SET OF4 WOODLAND , ROUND SN..."
13,0.118908,(WOODLAND CHARLOTTE BAG)
14,0.109162,(SPACEBOY LUNCH BOX )
26,0.103314,(PLASTERS IN TIN SPACEBOY)
1,0.097466,(JAM MAKING SET PRINTED)


FP-Growth findet genau dieselben 59 Itemsets mit denselben Support-Werten wie Apriori. Nur die Reihenfolge der Zeilen (der Index links) unterscheidet sich. Der Unterschied zwischen den Algorithmen liegt nicht im Ergebnis, sondern im Rechenweg: FP-Growth liest den Datensatz nur zweimal, speichert ihn komprimiert in einer Baumstruktur (*FP-Tree*) und erzeugt keine Kandidaten-Itemsets, die anschließend gezählt werden müssen. Bei großen Datensätzen ist das deutlich schneller.

### Schritt 2: Finde starke Association Rules

In [22]:
# Berechne die Association Rules (Confidence mindestens 40 %)
rules_fp = association_rules(frq_items_fp, metric='confidence', min_threshold=.4)

rules_fp[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False)

,antecedents,consequents,support,confidence,lift
8,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.058480,0.857143,7.208431
1,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.122807,0.818182,3.440387
14,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.052632,0.729730,5.849240
7,(SPACEBOY LUNCH BOX ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.068226,0.625000,2.628074
5,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.072125,0.578125,4.298234
10,(PLASTERS IN TIN WOODLAND ANIMALS),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.072125,0.536232,2.254811
4,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.536232,4.298234
12,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.054581,0.528302,3.927810
0,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.122807,0.516393,3.440387
6,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.060429,0.508197,2.136926


In [23]:
# Zeige die Ergebnisse geordnet nach 'lift'
rules_fp[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)

,antecedents,consequents,support,confidence,lift
9,(WOODLAND CHARLOTTE BAG),(RED RETROSPOT CHARLOTTE BAG),0.058480,0.491803,7.208431
8,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.058480,0.857143,7.208431
13,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN STRONGMAN),0.052632,0.421875,5.849240
14,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.052632,0.729730,5.849240
4,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.536232,4.298234
5,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.072125,0.578125,4.298234
11,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN SPACEBOY),0.054581,0.405797,3.927810
12,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.054581,0.528302,3.927810
0,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.122807,0.516393,3.440387
1,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.122807,0.818182,3.440387


Auch die Association Rules sind identisch mit den Ergebnissen von Apriori, nur die Nummerierung der Zeilen ist anders.

## Multilevel Association Rules

Bisher haben wir einzelne Produkte betrachtet. Produkte lassen sich aber auch in einer *Konzepthierarchie* zusammenfassen, z. B. „WOODLAND CHARLOTTE BAG“ → Taschen → Haushaltswaren. Auf höheren Ebenen der Hierarchie sind Muster häufiger, aber auch allgemeiner. *Multilevel Association Rules* suchen nach Regeln auf mehreren Ebenen der Hierarchie und auch zwischen den Ebenen (Han et al., 2023).

Unser Datensatz enthält keine Produktkategorien. Wir bauen deshalb eine einfache Hierarchie mit zwei Ebenen selbst: Produkt → Produkttyp.

### Eine Konzepthierarchie aus Schlüsselwörtern

In [24]:
# Schlüsselwörter in der Produktbezeichnung und die Kategorie, zu der sie gehören
# Die Reihenfolge ist wichtig: Das erste passende Schlüsselwort entscheidet
keywords = [
    ("LUNCH BOX", "Brotdosen"), ("SNACK BOX", "Brotdosen"),
    ("BAG", "Taschen"),
    ("PLASTERS", "Pflaster"),
    ("CAKE", "Backen"), ("BAKING", "Backen"), ("JAM", "Backen"),
    ("MUG", "Geschirr"), ("CUP", "Geschirr"), ("PLATE", "Geschirr"), ("BOWL", "Geschirr"),
    ("NAPKIN", "Geschirr"), ("TEA SET", "Geschirr"), ("CUTLERY", "Geschirr"), ("BEAKER", "Geschirr"),
    ("LIGHT", "Lichter"), ("LANTERN", "Lichter"), ("CANDLE", "Lichter"),
    ("CLOCK", "Uhren"),
    ("CARD", "Papeterie"), ("WRAP", "Papeterie"), ("NOTEBOOK", "Papeterie"), ("PENCIL", "Papeterie"),
    ("CHRISTMAS", "Weihnachten"),
]

# Ordne einem Produkt die Kategorie des ersten passenden Schlüsselworts zu
def find_category(description):
    for keyword, category in keywords:
        if keyword in description:
            return category
    return None # Kein Schlüsselwort passt

# Wende die Funktion auf alle Produkte (die Spalten von subset_df) an
categories = pd.Series({product: find_category(product) for product in subset_df.columns})

# Zähle die Produkte je Kategorie (dropna=False zählt auch die Produkte ohne Kategorie)
categories.value_counts(dropna=False)

None           1076
Papeterie       154
Geschirr        140
Taschen         115
Lichter         105
Backen           95
Weihnachten      60
Uhren            24
Brotdosen        11
Pflaster          6
Name: count, dtype: int64

Die Schlüsselwörter ordnen 710 der 1.786 Produkte einer von neun Kategorien zu. Die übrigen 1.076 Produkte (`None`) passen zu keinem Schlüsselwort und kommen auf der Ebene der Kategorien nicht vor. Da die zugeordneten Produkte häufiger gekauft werden, decken sie trotzdem rund die Hälfte aller gekauften Positionen ab. Die Zuordnung über Teilwörter ist grob: So landet „CREAM CUPID HEARTS COAT HANGER“ wegen „CUP“ bei Geschirr und „TEA BAG PLATE“ wegen „BAG“ bei Taschen. In der Praxis würde man statt Schlüsselwörtern die Produkthierarchie aus dem Warenwirtschaftssystem verwenden.

In [25]:
# Beispiel: alle Produkte der Kategorie "Pflaster"
categories[categories == "Pflaster"]

PLASTERS IN TIN CIRCUS PARADE       Pflaster
PLASTERS IN TIN SKULLS              Pflaster
PLASTERS IN TIN SPACEBOY            Pflaster
PLASTERS IN TIN STRONGMAN           Pflaster
PLASTERS IN TIN VINTAGE PAISLEY     Pflaster
PLASTERS IN TIN WOODLAND ANIMALS    Pflaster
dtype: object

Die sechs Pflaster-Dosen unterscheiden sich nur im Design. Auf der höheren Ebene der Hierarchie zählt nur noch, *dass* eine Rechnung Pflaster enthält, aber nicht mehr, welche.

### Transaktionen auf der Ebene der Kategorien

In [26]:
# Behalte nur die Produkte, die eine Kategorie haben
categorized = categories.dropna()

# Erstelle eine Spalte pro Kategorie: True, wenn die Rechnung mindestens ein Produkt dieser Kategorie enthält
category_df = pd.DataFrame(index=subset_df.index)
for category in sorted(categorized.unique()):
    products = categorized[categorized == category].index # Alle Produkte der Kategorie
    category_df[category] = subset_df[products].any(axis=1) # any: mindestens ein True pro Zeile

category_df

,Backen,Brotdosen,Geschirr,Lichter,Papeterie,Pflaster,Taschen,Uhren,Weihnachten
InvoiceNo,,,,,,,,,
536527,True,False,True,True,False,False,True,False,False
536840,True,False,False,False,False,False,False,False,False
536861,False,False,True,False,False,False,False,False,False
536967,False,False,False,False,False,False,True,False,False
536983,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...
581266,False,True,True,False,True,False,True,False,True
581494,False,True,False,True,True,True,False,True,False
581570,False,True,True,False,False,False,True,False,True


`category_df` hat dieselbe Struktur wie `subset_df`: eine Zeile pro Rechnung, aber nur noch eine Spalte pro Kategorie statt pro Produkt. `any(axis=1)` prüft für jede Rechnung, ob mindestens eines der Produkte der Kategorie gekauft wurde.

In [27]:
# Support jeder Kategorie: Anteil der Rechnungen, die die Kategorie enthalten
category_df.mean().sort_values(ascending=False)

Backen         0.555556
Taschen        0.499025
Geschirr       0.477583
Papeterie      0.387914
Lichter        0.352827
Brotdosen      0.348928
Pflaster       0.235867
Weihnachten    0.150097
Uhren          0.132554
dtype: float64

Die Kategorien kommen viel häufiger vor als einzelne Produkte: Backzubehör steckt in 56 % der Rechnungen, Taschen in 50 %. Das häufigste einzelne Produkt, die Woodland-Snackbox, kommt dagegen nur auf 24 %. Je höher die Ebene in der Hierarchie, desto höher der Support.

### Apriori mit reduziertem Support

Verwenden wir auf allen Ebenen denselben minimalen Support (*uniform support*), gibt es ein Dilemma: Ein niedriger Wert wie 5 % ist auf der Ebene der Kategorien fast immer erfüllt und erzeugt sehr viele, wenig interessante Itemsets. Ein hoher Wert wie 20 % würde auf der Ebene der Produkte dagegen fast alle Muster verwerfen. Deshalb setzt man den minimalen Support auf den unteren Ebenen niedriger an als auf den oberen (*reduced support*). Für die Produkte haben wir oben 5 % verwendet, für die Kategorien wählen wir 20 %.

In [28]:
# Finde häufige Itemsets auf der Ebene der Kategorien (minimaler Support 20 %)
frq_cat = apriori(category_df, min_support=0.2, use_colnames=True)

# Berechne die Association Rules (Confidence mindestens 50 %)
rules_cat = association_rules(frq_cat, metric='confidence', min_threshold=.5)

# Zeige die wichtigsten Spalten geordnet nach 'lift'
rules_cat[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(['lift'], ascending=[False])

,antecedents,consequents,support,confidence,lift
22,"(Taschen, Backen)",(Geschirr),0.224172,0.766667,1.605306
25,"(Papeterie, Taschen)",(Geschirr),0.200780,0.762963,1.597551
26,"(Geschirr, Taschen)",(Papeterie),0.200780,0.613095,1.580492
27,(Papeterie),"(Geschirr, Taschen)",0.200780,0.517588,1.580492
24,"(Papeterie, Geschirr)",(Taschen),0.200780,0.762963,1.528906
9,(Papeterie),(Brotdosen),0.202729,0.522613,1.497768
10,(Brotdosen),(Papeterie),0.202729,0.581006,1.497768
8,(Geschirr),(Brotdosen),0.249513,0.522449,1.497298
7,(Brotdosen),(Geschirr),0.249513,0.715084,1.497298
11,(Brotdosen),(Taschen),0.259259,0.743017,1.488936


Auf der Ebene der Kategorien finden wir 28 Regeln. Der Lift liegt zwischen 1,05 und 1,61 und damit deutlich niedriger als auf der Ebene der Produkte (bis 7,2). Die stärksten Regeln auf Produktebene verbanden Varianten desselben Produkttyps, etwa zwei Charlotte Bags oder zwei Pflaster-Dosen. Diese Regeln verschwinden auf der höheren Ebene, weil beide Produkte zur selben Kategorie gehören.

Stattdessen zeigen sich Zusammenhänge zwischen Produkttypen: Wer Brotdosen kauft, kauft in 74 % der Fälle auch Taschen und in 72 % auch Geschirr. Wer Backzubehör und Taschen kauft, kauft in 77 % der Fälle auch Geschirr.

### Regeln zwischen den Ebenen (cross-level)

Regeln können auch Ebenen verbinden, z. B. „wer die Pflaster-Dose im Spaceboy-Design kauft, kauft auch eine Brotdose“. Dafür fügen wir die Produkte und die Kategorien zu einem Datensatz zusammen.

Dabei entsteht ein Problem: Jede Rechnung, die ein Produkt enthält, enthält automatisch auch dessen Kategorie. Regeln wie „PLASTERS IN TIN SPACEBOY → Pflaster“ haben deshalb immer eine Confidence von 100 % und sagen nichts aus. Wir entfernen alle Itemsets, die ein Produkt zusammen mit seiner eigenen Kategorie enthalten.

In [29]:
# Füge Produkte und Kategorien zu einem Datensatz zusammen (eine Spalte pro Produkt und pro Kategorie)
mixed_df = pd.concat([subset_df, category_df], axis=1)

# Finde häufige Itemsets (minimaler Support 5 % wie bei den Produkten)
frq_mixed = apriori(mixed_df, min_support=0.05, use_colnames=True)

# Prüfe, ob ein Itemset ein Produkt zusammen mit seiner eigenen Kategorie enthält
def contains_own_category(itemset):
    for item in itemset:
        if item in categorized.index and categorized[item] in itemset:
            return True
    return False

# Entferne diese Itemsets (~ bedeutet 'nicht')
frq_mixed = frq_mixed[~frq_mixed['itemsets'].apply(contains_own_category)]

In [30]:
# Berechne die Association Rules (Confidence mindestens 50 %)
rules_mixed = association_rules(frq_mixed, metric='confidence', min_threshold=.5)

# Prüfe, ob ein Itemset aus genau einem Produkt bzw. genau einer Kategorie besteht
def is_single_product(itemset):
    return len(itemset) == 1 and list(itemset)[0] in subset_df.columns

def is_single_category(itemset):
    return len(itemset) == 1 and list(itemset)[0] in category_df.columns

# Behalte nur Regeln der Form "ein Produkt -> eine Kategorie"
cross_rules = rules_mixed[rules_mixed['antecedents'].apply(is_single_product)
                          & rules_mixed['consequents'].apply(is_single_category)]

cross_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(['lift'], ascending=[False])

,antecedents,consequents,support,confidence,lift
14,(LUNCH BAG APPLE DESIGN),(Brotdosen),0.052632,0.843750,2.418122
79,(WOODLAND PARTY BAG + STICKER SET),(Brotdosen),0.050682,0.812500,2.328561
29,(PLASTERS IN TIN SPACEBOY),(Brotdosen),0.076023,0.735849,2.108886
33,(PLASTERS IN TIN STRONGMAN),(Brotdosen),0.052632,0.729730,2.091348
22,(PLASTERS IN TIN CIRCUS PARADE ),(Brotdosen),0.087719,0.703125,2.015101
...,...,...,...,...,...
21,(PLASTERS IN TIN CIRCUS PARADE ),(Backen),0.076023,0.609375,1.096875
64,(ROUND SNACK BOXES SET OF4 WOODLAND ),(Backen),0.136452,0.573770,1.032787
58,(ROUND SNACK BOXES SET OF 4 FRUITS ),(Backen),0.081871,0.545455,0.981818
12,(JUMBO BAG WOODLAND ANIMALS),(Backen),0.052632,0.540000,0.972000


Es bleiben 69 Regeln von einem Produkt zu einer Kategorie. Die stärkste: 84 % der Rechnungen mit der „LUNCH BAG APPLE DESIGN“ enthalten auch eine Brotdose (Lift 2,4). Auch drei der Pflaster-Dosen (Spaceboy, Strongman, Circus Parade) werden oft zusammen mit Brotdosen gekauft (Confidence 70 bis 74 %, Lift rund 2). Am Ende der Liste stehen drei Regeln mit einem Lift unter 1: Trotz einer Confidence über 50 % werden diese Produkte *seltener* mit der Kategorie gekauft, als bei Unabhängigkeit zu erwarten wäre.

### Redundante Regeln

Viele Regeln zwischen den Ebenen sind *redundant*: Sie folgen bereits aus einer allgemeineren Regel. Han et al. (2023) nennen als Beispiel die Regeln „Milch → Weißbrot“ (Confidence 70 %) und „fettarme Milch → Weißbrot“ (Confidence 72 %). Die zweite Regel enthält kaum neue Information, denn ihre Confidence ist fast genauso hoch, wie es die übergeordnete Regel erwarten lässt.

Wir vergleichen deshalb jede Regel „Produkt → Kategorie“ mit der übergeordneten Regel „Kategorie des Produkts → Kategorie“. Liegen die beiden Confidence-Werte nahe beieinander (hier: weniger als 5 Prozentpunkte auseinander), ist die Regel redundant. Han et al. vergleichen neben der Confidence auch den Support mit dem erwarteten Wert. Wir vereinfachen hier und betrachten nur die Confidence, und die Grenze von 5 Prozentpunkten ist eine willkürliche Wahl.

In [31]:
# Berechne die Confidence der übergeordneten Regel "Kategorie des Produkts -> Kategorie"
def ancestor_confidence(rule):
    product = list(rule['antecedents'])[0]
    target = list(rule['consequents'])[0]
    if product not in categorized.index:
        return None # Das Produkt hat keine Kategorie, also gibt es keine übergeordnete Regel
    ancestor = categorized[product]
    support_both = (category_df[ancestor] & category_df[target]).mean() # Support von Kategorie und Ziel gemeinsam
    return support_both / category_df[ancestor].mean() # Confidence = Support(beide) / Support(Kategorie)

# Füge die Confidence der übergeordneten Regel als neue Spalte hinzu (axis=1: Funktion pro Zeile)
cross_rules = cross_rules.assign(ancestor_confidence = cross_rules.apply(ancestor_confidence, axis=1))

# Markiere Regeln als redundant, wenn die beiden Confidence-Werte weniger als 5 Prozentpunkte auseinanderliegen
cross_rules = cross_rules.assign(redundant = (cross_rules['confidence'] - cross_rules['ancestor_confidence']).abs() < 0.05)

cross_rules[['antecedents', 'consequents', 'confidence', 'ancestor_confidence', 'redundant']].sort_values(['confidence'], ascending=[False])

,antecedents,consequents,confidence,ancestor_confidence,redundant
4,(ALARM CLOCK BAKELIKE PINK),(Backen),0.891892,0.808824,False
43,(RED RETROSPOT CUP),(Taschen),0.843750,0.685714,False
14,(LUNCH BAG APPLE DESIGN),(Brotdosen),0.843750,0.519531,False
79,(WOODLAND PARTY BAG + STICKER SET),(Brotdosen),0.812500,0.519531,False
44,(RED RETROSPOT MINI CASES),(Geschirr),0.800000,NaN,False
...,...,...,...,...,...
24,(PLASTERS IN TIN CIRCUS PARADE ),(Lichter),0.546875,0.528926,True
58,(ROUND SNACK BOXES SET OF 4 FRUITS ),(Backen),0.545455,0.586592,True
12,(JUMBO BAG WOODLAND ANIMALS),(Backen),0.540000,0.585938,True
52,(REGENCY CAKESTAND 3 TIER),(Lichter),0.514706,0.414035,False


29 der 69 Regeln sind redundant. Ein Beispiel: 65 % der Rechnungen mit der „PLASTERS IN TIN WOODLAND ANIMALS“ enthalten eine Brotdose. Bei allen Rechnungen mit Pflastern sind es 63 %. Die Regel sagt also nichts, was nicht schon „Pflaster → Brotdosen“ sagt.

Interessant sind dagegen Regeln, die deutlich von der übergeordneten Regel abweichen: Mit der „LUNCH BAG APPLE DESIGN“ werden in 84 % der Fälle Brotdosen gekauft, mit Taschen allgemein nur in 52 %. Lunch Bags und Brotdosen gehören offenbar zusammen, was in der Kategorie „Taschen“ untergeht. Ein solches Ergebnis kann ein Anlass sein, die Hierarchie zu verfeinern, etwa mit einer eigenen Kategorie für Lunch Bags.

Für 10 Regeln gibt es keine übergeordnete Regel (`NaN`), weil das Produkt keiner Kategorie zugeordnet ist, etwa „RED RETROSPOT MINI CASES“. Diese Regeln werden nicht als redundant markiert.

## Quellen

* Han, J., Pei, J. und Tong, H. (2023): Data Mining: Concepts and Techniques. 4. Auflage. Morgan Kaufmann.
* Kelleher, J. D., Mac Namee, B. und D'Arcy, A. (2020): Fundamentals of Machine Learning for Predictive Data Analytics: Algorithms, Worked Examples, and Case Studies. 2. Auflage. MIT Press.
* https://developer.ibm.com/tutorials/awb-implementing-apriori-algorithm-python/
* https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/